# broadcast-initial-weights — ex1: broadcast rank-0 weights to all ranks at start

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `broadcast-initial-weights`. Running the final beacon cell reports progress against the `Distributed: broadcast initial weights` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: broadcast initial weights` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`broadcast-initial-weights`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcast-initial-weights"
DD_SUBTOPIC = "Distributed: broadcast initial weights"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch.distributed quick refresher

PyTorch's collective-communication library (`torch.distributed`, aliased `dist`) lets multiple processes coordinate over tensors. Each rank runs the same function in its own process; collectives operate in-place on tensors of identical shape across ranks.

**Backends.** `'nccl'` for multi-GPU (ARENA's setup), `'gloo'` for CPU (what these drills use — Colab CPU runtimes have no GPUs).

**Launch pattern.** Each test uses `mp.get_context('fork').Process` so worker fns defined in a notebook cell are picklable. Workers init the group, do their work, push results onto a `manager.Queue`, then destroy the group.

### This drill's atom: broadcast initial weights at training start
Every rank constructs its own `nn.Module` — but stochastic init means each rank ends up with DIFFERENT random weights. For data-parallel training to be valid, all ranks must start from the SAME parameters. The standard pattern:
```python
model = MyModel()  # each rank has random weights
for p in model.parameters():
    dist.broadcast(p.data, src=0)
# now every rank's params == rank 0's params
```
After this, `loss.backward()` + grad-sync keep them in lockstep forever.

### Exercise 1 — broadcast rank-0 weights to all ranks at start

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply `dist.broadcast(p.data, src=0)` over every parameter to synchronize all ranks to rank 0's initial weights at the start of data-parallel training.
> Keywords: broadcast, DDP-init, parameter-sync, rank-0-source
> ```

**KCs targeted:** `loop-parameters-broadcast`, `broadcast-src-rank-0`

Implement `ex1_broadcast_init_worker(rank, world_size, port, out_queue)`. Each rank:

1. Inits `gloo`.
2. Builds a fresh model whose params depend on `rank` (this is the test fixture — each rank gets *different* initial weights to expose the bug if broadcast is skipped):
   ```python
   model = t.nn.Linear(2, 2, bias=False)
   with t.no_grad():
       model.weight.fill_(float(rank + 1))
   ```
   So rank 0 has weights `[[1,1],[1,1]]`, rank 1 has `[[2,2],[2,2]]`.
3. **Broadcast rank-0 weights to every other rank:**
   ```python
   for p in model.parameters():
       dist.broadcast(p.data, src=0)
   ```
4. Pushes `(rank, model.weight.detach().flatten().tolist())` onto `out_queue`.
5. Destroys process group.

**Expected.** Pre-broadcast: rank 0 = `[1,1,1,1]`, rank 1 = `[2,2,2,2]`. Post-broadcast: BOTH ranks hold `[1,1,1,1]` (rank 0's values won).

In [ ]:
import os, datetime
import torch as t
import torch.distributed as dist

def ex1_broadcast_init_worker(rank, world_size, port, out_queue):
    """Init gloo, broadcast rank-0 weights to all ranks, queue result."""
    raise NotImplementedError()


def _test_ex1():
    import os as _os
    import datetime as _dt
    import torch.distributed as _dist
    import torch.multiprocessing as _mp

    def _dd_run_workers(worker_fn, world_size, port, *extra_args, timeout=30):
        """Spawn `world_size` fork-context procs, return list of exitcodes."""
        ctx = _mp.get_context('fork')
        procs = []
        for rank in range(world_size):
            p = ctx.Process(target=worker_fn, args=(rank, world_size, port, *extra_args))
            p.start()
            procs.append(p)
        for p in procs:
            p.join(timeout=timeout)
        codes = [p.exitcode for p in procs]
        for p in procs:
            if p.is_alive():
                p.terminate()
        return codes

    manager = _mp.Manager()
    q = manager.Queue()
    codes = _dd_run_workers(ex1_broadcast_init_worker, 2, 29610, q)
    assert codes == [0, 0], f'workers failed: {codes}'

    results = {}
    while not q.empty():
        rank, weights = q.get()
        results[rank] = weights
    assert set(results.keys()) == {0, 1}, f'expected ranks {{0,1}}, got {set(results.keys())}'
    # Both ranks should hold rank-0's original weights (all 1.0).
    for rank, w in results.items():
        assert len(w) == 4, f'rank {rank}: expected 4 weights, got {len(w)}'
        for v in w:
            assert abs(v - 1.0) < 1e-6, f'rank {rank}: expected 1.0, got {v} (broadcast skipped?)'

    # 3-rank case — every rank should still converge to rank-0's [1,1,1,1].
    q3 = manager.Queue()
    codes3 = _dd_run_workers(ex1_broadcast_init_worker, 3, 29611, q3)
    assert codes3 == [0, 0, 0], f'3-rank workers failed: {codes3}'
    results3 = {}
    while not q3.empty():
        rank, w = q3.get()
        results3[rank] = w
    assert set(results3.keys()) == {0, 1, 2}
    for rank, w in results3.items():
        for v in w:
            assert abs(v - 1.0) < 1e-6, f'3-rank case, rank {rank}: got {v}'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_broadcast_init_worker(rank, world_size, port, out_queue):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(backend='gloo', rank=rank, world_size=world_size,
                            timeout=datetime.timedelta(seconds=20))
    model = t.nn.Linear(2, 2, bias=False)
    with t.no_grad():
        model.weight.fill_(float(rank + 1))
    # Sync initial weights — every rank now mirrors rank 0.
    for p in model.parameters():
        dist.broadcast(p.data, src=0)
    out_queue.put((rank, model.weight.detach().flatten().tolist()))
    dist.destroy_process_group()
```

**Why `p.data` not `p`.** `dist.broadcast` operates on a `Tensor`, but `nn.Parameter` is a `Tensor` subclass. Using `p.data` strips the autograd machinery and gives you the raw storage — broadcast writes in-place into that storage. Using `p` directly works in modern PyTorch but is conventionally avoided to make the in-place mutation explicit.

**Why broadcast not all_reduce.** `all_reduce` SUMS across ranks — you'd get `(1+2)/2 = 1.5` weights, not rank 0's `1.0`. Broadcast picks ONE source rank and copies its tensor to all others.

**Alternative: seed every rank identically.** `t.manual_seed(0)` before the model constructor *would* give matching weights — but only if every rank uses the same code path. Broadcast is the robust pattern: it works even when ranks load different checkpoints, use different init strategies, or boot from non-deterministic ops.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()